# Project: Carbon Footprint Prediction (India)
## Notebook 01: Initial Data Consolidation

[![Author](https://img.shields.io/badge/Author-Prakash%20Ukhalkar-blue.svg)](https://github.com/prakash-ukhalkar) [![GitHub Repository](https://img.shields.io/badge/GitHub-Repo-lightgrey)](https://github.com/prakash-ukhalkar/India-Agri-Carbon-Prediction) [![Python](https://img.shields.io/badge/Python-3.10%2B-blue)](https://www.python.org/) [![Pandas](https://img.shields.io/badge/Pandas-Latest-green)](https://pandas.pydata.org/) [![NumPy](https://img.shields.io/badge/NumPy-Latest-lightblue)](https://numpy.org/) [![statsmodels](https://img.shields.io/badge/statsmodels-Latest-orange)](https://www.statsmodels.org/) [![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

---

**Objective:** Merge 32 raw CSVs/Excels into 3 thematic Master Research Files to simplify the ML pipeline.

**Introduction:** To manage the high volume of raw data (32 files), this notebook automates the consolidation process. We will group related datasets, standardize the "Year" column across all entries, and preserve every original feature. This ensures a clean workspace without losing the "long-tail" details required for high-impact research.

## Step 1: Programmatic Data Acquisition & File Mapping

**Programmatic Data Integration via APIs**
To ensure complete reproducibility and maintain publication-grade standards for MethodsX, the data ingestion pipeline includes a programmatic data acquisition module (`scripts/acquire_data_api.py`). This script queries:
1. **World Bank API (WDI)**: Programmatically downloads agricultural land area (`AG.LND.AGRI.K2`), fertilizer consumption (`AG.CON.FERT.ZS`), crop production indexes (`AG.PRD.CROP.XD`), population, and GDP.
2. **Open-Meteo Historical Weather API**: Retrieves daily precipitation and temperature for India (gridded ERA5 reanalysis data) and aggregates them into annual climate features (`Annual_Rainfall` and `Temperature`).
3. **FAOSTAT and Local Data**: Merges the API-fetched data with local raw emissions and activity datasets.

**Rationale for moving from 32 files to 3 thematic masters:** 
Managing 32 separate files introduces significant overhead during the feature engineering and modeling phases. By mapping them into three core thematic areas (Emissions, Energy & Activity, Land & Water Resources), we streamline our pipeline while maintaining the original granularity of the data. The master files are programmatically updated using the API module.

In [1]:
import pandas as pd
import os
import glob
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Define base directories
RAW_DATA_DIR = '../data/raw/'
PROCESSED_DATA_DIR = '../data/processed/'

# Ensure processed directory exists
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

# Dictionary mapping 32 files into categories
file_mapping = {
    'Emissions': glob.glob(os.path.join(RAW_DATA_DIR, 'Emission_Data/**/*.*'), recursive=True),
    'Energy': glob.glob(os.path.join(RAW_DATA_DIR, 'Energy_and_Resource_Consumption_Data/**/*.*'), recursive=True) + glob.glob(os.path.join(RAW_DATA_DIR, 'Farm_Activity_Data/**/*.*'), recursive=True),
    'Resources': glob.glob(os.path.join(RAW_DATA_DIR, 'Environment_and_Soil_Data/**/*.*'), recursive=True)
}

# Filter only tabular files
file_mapping = {k: [f for f in v if f.endswith(('.csv', '.xls', '.xlsx'))] for k, v in file_mapping.items()}

print("Total files mapped:")
for category, files in file_mapping.items():
    print(f"{category}: {len(files)} files")

Total files mapped:
Emissions: 3 files
Energy: 13 files
Resources: 15 files


## Step 2: The "Standardizer" Function
Handling the "Fiscal Year" (e.g., 1995-96) vs "Calendar Year" (e.g., 1995) to ensure all 32 files speak the same temporal language. We extract the starting year to a uniform integer.

In [2]:
import re
import pandas as pd

def clean_year_column(df):
    """
    Detects and cleans year columns ('Fiscal_Year', 'StartYear', 'Year').
    Converts formats like '1995-96' or '1995' to uniform integer Year.
    """
    year_cols = ['Fiscal_Year', 'StartYear', 'Year', 'Year Code', 'period']
    
    # Flatten MultiIndex columns if any
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ['_'.join(map(str, col)).strip() for col in df.columns.values]
        
    for col in df.columns:
        col_name = str(col)
        if any(term.lower() in col_name.lower() for term in year_cols):
            # Extract the 4-digit year
            def extract_year(val):
                val_str = str(val)
                match = re.search(r'\b(19\d{2}|20\d{2})\b', val_str)
                if match:
                    return int(match.group(1))
                return pd.NA
                
            df['Year_Standardized'] = df[col].apply(extract_year)
            df = df.dropna(subset=['Year_Standardized'])
            df['Year_Standardized'] = df['Year_Standardized'].astype(int)
            break # process only the first detected year column to standardize
            
    # If no standard year could be extracted, log a warning
    if 'Year_Standardized' not in df.columns:
        pass
        
    return df

def load_tabular_data(filepath):
    import io
    if filepath.endswith('.csv'):
        return pd.read_csv(filepath)
    try:
        return pd.read_excel(filepath)
    except Exception as e:
        # Fallback for HTML tables disguised as .xls
        try:
            with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read()
            if '<table' in content.lower():
                dfs = pd.read_html(io.StringIO(content))
                if dfs:
                    return dfs[0]
        except Exception:
            pass
        # Fallback for TSVs disguised as .xls
        try:
            return pd.read_csv(filepath, sep='\t')
        except Exception:
            print(f"Failed to load {filepath}: {e}")
            return pd.DataFrame()


## Step 3: Creating the Emissions Master
Consolidation of FAOSTAT Crop and Energy emissions. Creates a long-format panel, preserving elements like 'Item' and 'Element'.

In [3]:
emission_dfs = []

for filepath in file_mapping['Emissions']:
    df = load_tabular_data(filepath)
    if df.empty:
        continue
    df = clean_year_column(df)
    df['Source_File'] = Path(filepath).name
    emission_dfs.append(df)

if emission_dfs:
    # Creating a long-format panel by concatenating all emission files
    master_emissions_panel = pd.concat(emission_dfs, ignore_index=True)
    
    output_path = os.path.join(PROCESSED_DATA_DIR, 'master_emissions_panel.csv')
    master_emissions_panel.to_csv(output_path, index=False)
    print(f"Emissions Master saved to {output_path} with shape: {master_emissions_panel.shape}")
else:
    print("No Emissions files found.")

Emissions Master saved to ../data/processed/master_emissions_panel.csv with shape: (3660, 20)


## Step 4: Creating the Energy & Activity Master
Merging electricity, renewable shares, and fuel use. Ensuring the CAGR-forecasted years (2023-2025) are flagged but included.

In [4]:
energy_dfs = []

# Combining Electricity_Agri_Use, RenewableEnergy_Agri, etc.
for filepath in file_mapping['Energy']:
    df = load_tabular_data(filepath)
    if df.empty:
        continue
    df = clean_year_column(df)
    
    if 'Year_Standardized' in df.columns:
        # Flag forecasted years
        df['Is_Forecasted'] = df['Year_Standardized'].apply(lambda y: True if y >= 2023 else False)
        
    df['Source_File'] = Path(filepath).name
    energy_dfs.append(df)

if energy_dfs:
    energy_dfs = [df for df in energy_dfs if 'Year_Standardized' in df.columns]
    
    if energy_dfs:
        # Prevent MemoryError by ensuring unique keys per dataframe before merging
        # Aggregate any duplicate year rows in each individual DF first
        agg_dfs = []
        for d in energy_dfs:
            # Group by year and take first non-null value for each column to flatten
            d_agg = d.groupby('Year_Standardized', as_index=False).first()
            agg_dfs.append(d_agg)
            
        master_energy_activity = agg_dfs[0]
        for df in agg_dfs[1:]:
            overlap_cols = [c for c in master_energy_activity.columns if c in df.columns and c != 'Year_Standardized']
            df = df.drop(columns=overlap_cols, errors='ignore')
            master_energy_activity = pd.merge(master_energy_activity, df, on='Year_Standardized', how='outer')
        
        # Sort by year
        master_energy_activity = master_energy_activity.sort_values('Year_Standardized').reset_index(drop=True)
        
        output_path = os.path.join(PROCESSED_DATA_DIR, 'master_energy_activity.csv')
        master_energy_activity.to_csv(output_path, index=False)
        print(f"Energy & Activity Master saved to {output_path} with shape: {master_energy_activity.shape}")
else:
    print("No Energy & Activity files found.")

Energy & Activity Master saved to ../data/processed/master_energy_activity.csv with shape: (65, 317)


## Step 5: Creating the Land & Water Master
Integrating Irrigated/Rainfed areas and Water withdrawal proxies. Aligning them strictly by the Year index.

In [5]:
resource_dfs = []

for filepath in file_mapping['Resources']:
    # Integrating Irrigated_Land, Rainfed_Land, and India_Irrigation_Water
    if "Land" in filepath or "Irrigation" in filepath or "Water" in filepath:
        df = load_tabular_data(filepath)
        if df.empty:
            continue
        df = clean_year_column(df)
        if 'Year_Standardized' in df.columns:
            df['Source_File'] = Path(filepath).name
            resource_dfs.append(df)

if resource_dfs:
    # Aggregating strictly by Year index first to prevent MemoryError (Cartesian Products)
    agg_dfs = []
    for d in resource_dfs:
        d_agg = d.groupby('Year_Standardized', as_index=False).first()
        agg_dfs.append(d_agg)

    master_land_water_resource = agg_dfs[0]
    for df in agg_dfs[1:]:
        overlap_cols = [c for c in master_land_water_resource.columns if c in df.columns and c != 'Year_Standardized']
        df = df.drop(columns=overlap_cols, errors='ignore')
        master_land_water_resource = pd.merge(master_land_water_resource, df, on='Year_Standardized', how='outer')
        
    # Sorting strictly by the Year index
    master_land_water_resource = master_land_water_resource.sort_values(by='Year_Standardized').reset_index(drop=True)
    
    output_path = os.path.join(PROCESSED_DATA_DIR, 'master_land_water_resource.csv')
    master_land_water_resource.to_csv(output_path, index=False)
    print(f"Land & Water Master saved to {output_path} with shape: {master_land_water_resource.shape}")
else:
    print("No Land/Water associated files found.")

Land & Water Master saved to ../data/processed/master_land_water_resource.csv with shape: (65, 30)


## Step 6: Final Verification
Audit of the new files to verify outputs.

In [6]:
def audit_master_file(filepath, name):
    if not os.path.exists(filepath):
        print(f"{name} not found.")
        return
        
    df = pd.read_csv(filepath)
    print(f"=== {name} ===")
    print(f"Shape: {df.shape}")
    
    if 'Year_Standardized' in df.columns:
        years = sorted(df['Year_Standardized'].dropna().unique())
        print(f"Year Range: {min(years) if years else 'N/A'} to {max(years) if years else 'N/A'}")
        print(f"Unique Years Count: {len(years)}")
    print("-" * 30)

print("\n--- MASTER FILES AUDIT ---")
audit_master_file(os.path.join(PROCESSED_DATA_DIR, 'master_emissions_panel.csv'), "Emissions Master")
audit_master_file(os.path.join(PROCESSED_DATA_DIR, 'master_energy_activity.csv'), "Energy & Activity Master")
audit_master_file(os.path.join(PROCESSED_DATA_DIR, 'master_land_water_resource.csv'), "Land & Water Resource Master")


--- MASTER FILES AUDIT ---


=== Emissions Master ===
Shape: (3660, 20)
Year Range: 1961 to 2023
Unique Years Count: 63
------------------------------
=== Energy & Activity Master ===
Shape: (65, 317)
Year Range: 1961 to 2025
Unique Years Count: 65
------------------------------
=== Land & Water Resource Master ===
Shape: (65, 30)
Year Range: 1961 to 2025
Unique Years Count: 65
------------------------------


### Key Findings
Based on the initial data consolidation process:
- **Data Consolidation:** Successfully merged 32 disparate Excel and CSV files into 3 consolidated thematic master panels (Emissions, Energy & Activity, Land & Water Resources).
- **Data Standardization:** Temporal data was unified by extracting standard Calendar Years from assorted "Fiscal Year" and text-based formats, facilitating robust time-series alignment.
- **Dimensionality Preservation:** Original file-level granularity was maintained within a long-format panel while flattening and pivoting overlapping columns, ensuring no loss of detail for feature engineering.

---
### End of Notebook 01 — Initial Data Consolidation

**Outputs produced:**
- `../data/processed/master_emissions_panel.csv` — master emissions data
- `../data/processed/master_energy_activity.csv` — master energy and activity data
- `../data/processed/master_land_water_resource.csv` — master land and water resource data

**Next step → Notebook 02:** Feature Engineering & Target Alignment. Alignment of target variables, checking stationarity, and engineering lag/momentum features.

<div align="center"><sub>END OF NOTEBOOK 01</sub></div>